# Standalone Colab Phone/Watch Baselines

This notebook is self-contained: no repo clone, no local modules. Point it at your cached `artifacts/windows` folder in Google Drive and it will run the supervised `phone` and `watch` baselines sequentially on CUDA, with separate resumable output directories for each channel mode.

In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
print('Drive mounted')

In [ ]:
WINDOWS_ROOT = '/content/drive/MyDrive/cs286-project/artifacts/windows'
OUTPUT_ROOT = '/content/drive/MyDrive/cs286-project-runs'

CHANNEL_MODES = ['phone', 'watch']
RUN_NAMES = {
    'phone': 'phone-full',
    'watch': 'watch-full',
}
EXPERIMENT_NAMES = {
    'phone': 'phone_full',
    'watch': 'watch_full',
}
FORCE_FRESH_RUN = False

LABEL_FRACTION = 1.0
SEED = 42
BATCH_SIZE = 256
LR = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS = 40
PATIENCE = 7
GRAD_CLIP = 1.0
DROPOUT = 0.2
NUM_WORKERS = 2
SAVE_EVERY = 1

MAX_TRAIN_WINDOWS = None
MAX_VAL_WINDOWS = None
MAX_TEST_WINDOWS = None

print({
    'windows_root': WINDOWS_ROOT,
    'output_root': OUTPUT_ROOT,
    'channel_modes': CHANNEL_MODES,
    'run_names': RUN_NAMES,
    'experiment_names': EXPERIMENT_NAMES,
})

In [ ]:
import csv
import json
import pickle
import random
from dataclasses import dataclass
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def require_cuda() -> torch.device:
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA is not available. In Colab, switch the runtime to GPU.')
    device = torch.device('cuda')
    print('Using GPU:', torch.cuda.get_device_name(0))
    return device


@dataclass(frozen=True)
class CachedWindowSample:
    split: str
    raw_split: str
    subject_id: int
    activity_label: str
    occurrence_index: int
    offset_index: int
    center_s: float
    start_s: float
    end_s: float
    window_id: int
    label_idx: int
    x_fusion: np.ndarray
    x_phone: np.ndarray
    x_watch: np.ndarray


@dataclass(frozen=True)
class CachedSplitData:
    split: str
    samples: tuple[CachedWindowSample, ...]

    def __len__(self) -> int:
        return len(self.samples)


class WindowRepository:
    def __init__(self, windows_root: str | Path):
        self.windows_root = Path(windows_root)
        manifest_path = self.windows_root / 'manifest.json'
        if not manifest_path.exists():
            raise FileNotFoundError(f'Missing manifest: {manifest_path}')
        self.manifest = json.loads(manifest_path.read_text())

    def load_split(self, split: str) -> CachedSplitData:
        metadata_path = self.windows_root / split / 'metadata.csv'
        if not metadata_path.exists():
            raise FileNotFoundError(f'Missing metadata: {metadata_path}')

        rows = []
        with metadata_path.open('r', encoding='utf-8', newline='') as handle:
            reader = csv.DictReader(handle)
            for row in reader:
                rows.append({
                    'split': row['split'],
                    'raw_split': row['raw_split'],
                    'subject_id': int(row['subject_id']),
                    'activity_label': row['activity_label'],
                    'occurrence_index': int(row['occurrence_index']),
                    'offset_index': int(row['offset_index']),
                    'center_s': float(row['center_s']),
                    'start_s': float(row['start_s']),
                    'end_s': float(row['end_s']),
                    'window_id': int(row['window_id']),
                    'label_idx': int(row['label_idx']),
                })

        payloads = {}
        for chunk_path in sorted((self.windows_root / split).glob('data_chunk_*.pkl')):
            with chunk_path.open('rb') as handle:
                records = pickle.load(handle)
            for record in records:
                payloads[int(record['window_id'])] = np.asarray(record['x_fusion'], dtype=np.float32)

        samples = []
        for row in rows:
            payload = payloads[row['window_id']]
            samples.append(CachedWindowSample(
                split=row['split'],
                raw_split=row['raw_split'],
                subject_id=row['subject_id'],
                activity_label=row['activity_label'],
                occurrence_index=row['occurrence_index'],
                offset_index=row['offset_index'],
                center_s=row['center_s'],
                start_s=row['start_s'],
                end_s=row['end_s'],
                window_id=row['window_id'],
                label_idx=row['label_idx'],
                x_fusion=payload.copy(),
                x_phone=payload[:6].copy(),
                x_watch=payload[6:].copy(),
            ))
        return CachedSplitData(split=split, samples=tuple(samples))


def limit_samples(samples, max_samples):
    return samples if max_samples is None else tuple(samples[:max_samples])


def select_labeled_subset(samples, label_fraction: float, seed: int):
    if label_fraction >= 1.0:
        return tuple(samples)
    if label_fraction <= 0.0:
        raise ValueError('label_fraction must be > 0')
    rng = random.Random(seed)
    by_subject = {}
    for sample in samples:
        by_subject.setdefault(sample.subject_id, []).append(sample)
    selected = []
    for subject_id in sorted(by_subject):
        subject_samples = list(by_subject[subject_id])
        rng.shuffle(subject_samples)
        keep = max(1, int(round(len(subject_samples) * label_fraction)))
        selected.extend(subject_samples[:keep])
    selected.sort(key=lambda sample: (sample.subject_id, sample.window_id))
    return tuple(selected)


def build_supervised_arrays(split_data: CachedSplitData, channel_mode: str):
    if channel_mode == 'fusion':
        x = np.stack([sample.x_fusion for sample in split_data.samples]).astype(np.float32)
    elif channel_mode == 'phone':
        x = np.stack([sample.x_phone for sample in split_data.samples]).astype(np.float32)
    elif channel_mode == 'watch':
        x = np.stack([sample.x_watch for sample in split_data.samples]).astype(np.float32)
    else:
        raise ValueError(f'Unsupported channel_mode: {channel_mode}')
    y = np.asarray([sample.label_idx for sample in split_data.samples], dtype=np.int64)
    subjects = np.asarray([sample.subject_id for sample in split_data.samples], dtype=np.int64)
    return x, y, subjects


def to_loader(x: np.ndarray, y: np.ndarray, *, batch_size: int, shuffle: bool, num_workers: int) -> DataLoader:
    dataset = TensorDataset(torch.from_numpy(x), torch.from_numpy(y))
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=num_workers)


class TimeSeriesEncoder(nn.Module):
    output_dim = 256

    def __init__(self, in_channels: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(in_channels, 64, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.features(x).squeeze(-1)


class SupervisedHARModel(nn.Module):
    def __init__(self, in_channels: int, num_classes: int, dropout: float = 0.2):
        super().__init__()
        self.encoder = TimeSeriesEncoder(in_channels=in_channels)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.encoder.output_dim, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.encoder(x))


class EarlyStopper:
    def __init__(self, patience: int, mode: str = 'max'):
        self.patience = patience
        self.mode = mode
        self.best_score = None
        self.best_epoch = None
        self.bad_epochs = 0
        self.should_stop = False

    def update(self, score: float, epoch: int) -> bool:
        improved = False
        if self.best_score is None:
            improved = True
        elif self.mode == 'max' and score > self.best_score:
            improved = True
        elif self.mode == 'min' and score < self.best_score:
            improved = True
        if improved:
            self.best_score = score
            self.best_epoch = epoch
            self.bad_epochs = 0
            self.should_stop = False
            return True
        self.bad_epochs += 1
        self.should_stop = self.bad_epochs >= self.patience
        return False

    def state_dict(self) -> dict:
        return {
            'patience': self.patience,
            'mode': self.mode,
            'best_score': self.best_score,
            'best_epoch': self.best_epoch,
            'bad_epochs': self.bad_epochs,
            'should_stop': self.should_stop,
        }

    @classmethod
    def from_state_dict(cls, state: dict):
        stopper = cls(patience=int(state['patience']), mode=str(state['mode']))
        stopper.best_score = state.get('best_score')
        stopper.best_epoch = state.get('best_epoch')
        stopper.bad_epochs = int(state.get('bad_epochs', 0))
        stopper.should_stop = bool(state.get('should_stop', False))
        return stopper


def confusion_matrix_from_predictions(y_true, y_pred, num_classes: int) -> np.ndarray:
    matrix = np.zeros((num_classes, num_classes), dtype=np.int64)
    for truth, pred in zip(y_true, y_pred):
        matrix[int(truth), int(pred)] += 1
    return matrix


def macro_f1_from_confusion(matrix: np.ndarray, index_to_label: dict[int, str]):
    scores = []
    per_class_f1 = {}
    for class_index in range(matrix.shape[0]):
        tp = float(matrix[class_index, class_index])
        fp = float(matrix[:, class_index].sum() - tp)
        fn = float(matrix[class_index, :].sum() - tp)
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2.0 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
        per_class_f1[index_to_label[class_index]] = f1
        scores.append(f1)
    return float(sum(scores) / len(scores)) if scores else 0.0, per_class_f1


def per_subject_accuracy(y_true, y_pred, subject_ids):
    counts = {}
    for subject_id, truth, pred in zip(subject_ids, y_true, y_pred):
        key = int(subject_id)
        counts.setdefault(key, [0, 0])
        counts[key][0] += int(int(truth) == int(pred))
        counts[key][1] += 1
    return {
        subject_id: correct / total
        for subject_id, (correct, total) in sorted(counts.items())
        if total > 0
    }


def compute_metrics(y_true, y_pred, subject_ids, index_to_label, loss: float | None = None) -> dict:
    matrix = confusion_matrix_from_predictions(y_true, y_pred, num_classes=len(index_to_label))
    accuracy = float(matrix.diagonal().sum() / max(1, matrix.sum()))
    macro_f1, per_class_f1 = macro_f1_from_confusion(matrix, index_to_label)
    return {
        'loss': loss,
        'accuracy': accuracy,
        'macro_f1': macro_f1,
        'per_class_f1': per_class_f1,
        'per_subject_accuracy': per_subject_accuracy(y_true, y_pred, subject_ids),
        'confusion_matrix': matrix,
    }


def evaluate_classifier(model, loader, subjects, loss_fn, index_to_label, device) -> dict:
    model.eval()
    total_loss = 0.0
    total_count = 0
    all_true = []
    all_pred = []
    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            logits = model(x_batch)
            loss = loss_fn(logits, y_batch)
            total_loss += float(loss.item()) * len(x_batch)
            total_count += len(x_batch)
            predictions = logits.argmax(dim=1)
            all_true.extend(y_batch.cpu().tolist())
            all_pred.extend(predictions.cpu().tolist())
    return compute_metrics(
        all_true,
        all_pred,
        subjects.tolist(),
        index_to_label,
        loss=total_loss / max(1, total_count),
    )


def plot_confusion_matrix(matrix: np.ndarray, index_to_label: dict[int, str], title: str, output_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(10, 8))
    image = ax.imshow(matrix, cmap='Blues')
    ax.set_title(title)
    ax.set_xlabel('Predicted label')
    ax.set_ylabel('True label')
    ticks = np.arange(len(index_to_label))
    labels = [index_to_label[index] for index in ticks]
    ax.set_xticks(ticks)
    ax.set_xticklabels(labels)
    ax.set_yticks(ticks)
    ax.set_yticklabels(labels)
    fig.colorbar(image, ax=ax)
    fig.tight_layout()
    fig.savefig(output_path, dpi=150)
    plt.close(fig)


def write_subject_accuracy(path: Path, subject_scores: dict[int, float]) -> None:
    with path.open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=['subject_id', 'accuracy'])
        writer.writeheader()
        for subject_id, accuracy in subject_scores.items():
            writer.writerow({'subject_id': subject_id, 'accuracy': accuracy})


def save_checkpoint(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(payload, path)


def load_checkpoint(path: Path) -> dict:
    return torch.load(path, map_location='cpu', weights_only=False)


def metric_token(value: float) -> str:
    return f'{value:.4f}'.replace('.', 'p')


def checkpoint_metadata(experiment_name: str, channel_mode: str, train_split, val_split, test_split, windows_root: Path, checkpoint_role: str | None = None, checkpoint_epoch: int | None = None) -> dict:
    metadata = {
        'experiment_name': experiment_name,
        'stage': 'supervised',
        'created_at': 'colab',
        'manifest_path': str(windows_root / 'manifest.json'),
        'config': {
            'channel_mode': channel_mode,
            'label_fraction': LABEL_FRACTION,
            'batch_size': BATCH_SIZE,
            'learning_rate': LR,
            'weight_decay': WEIGHT_DECAY,
            'dropout': DROPOUT,
            'epochs_requested': EPOCHS,
            'early_stopping_patience': PATIENCE,
            'train_windows_used': len(train_split),
            'val_windows': len(val_split),
            'test_windows': len(test_split),
            'save_every': SAVE_EVERY,
            'seed': SEED,
        },
    }
    if checkpoint_role is not None:
        metadata['checkpoint_role'] = checkpoint_role
    if checkpoint_epoch is not None:
        metadata['checkpoint_epoch'] = checkpoint_epoch
    return metadata


def run_supervised_baseline(channel_mode: str, repository: WindowRepository, device: torch.device) -> dict:
    seed_everything(SEED)
    windows_root = Path(WINDOWS_ROOT)
    run_name = RUN_NAMES[channel_mode]
    experiment_name = EXPERIMENT_NAMES[channel_mode]
    output_dir = Path(OUTPUT_ROOT) / 'supervised' / run_name
    output_dir.mkdir(parents=True, exist_ok=True)
    latest_checkpoint_path = output_dir / 'latest_checkpoint.pt'

    train_split = repository.load_split('train')
    val_split = repository.load_split('val')
    test_split = repository.load_split('test')

    train_split = CachedSplitData('train', select_labeled_subset(train_split.samples, LABEL_FRACTION, SEED))
    train_split = CachedSplitData('train', tuple(limit_samples(train_split.samples, MAX_TRAIN_WINDOWS)))
    val_split = CachedSplitData('val', tuple(limit_samples(val_split.samples, MAX_VAL_WINDOWS)))
    test_split = CachedSplitData('test', tuple(limit_samples(test_split.samples, MAX_TEST_WINDOWS)))

    train_x, train_y, train_subjects = build_supervised_arrays(train_split, channel_mode)
    val_x, val_y, val_subjects = build_supervised_arrays(val_split, channel_mode)
    test_x, test_y, test_subjects = build_supervised_arrays(test_split, channel_mode)

    model = SupervisedHARModel(train_x.shape[1], len(repository.manifest['label_to_index']), dropout=DROPOUT).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.CrossEntropyLoss()

    train_loader = to_loader(train_x, train_y, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    val_loader = to_loader(val_x, val_y, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    test_loader = to_loader(test_x, test_y, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    early_stopper = EarlyStopper(PATIENCE, mode='max')
    best_state = None
    history = []
    start_epoch = 1

    if latest_checkpoint_path.exists() and not FORCE_FRESH_RUN:
        resume = load_checkpoint(latest_checkpoint_path)
        model.load_state_dict(resume['model_state_dict'])
        optimizer.load_state_dict(resume['optimizer_state_dict'])
        early_stopper = EarlyStopper.from_state_dict(resume['early_stopper_state_dict'])
        best_state = resume.get('best_model_state_dict')
        history = [dict(entry) for entry in resume.get('history', [])]
        start_epoch = int(resume['epoch']) + 1
        print(f'[{experiment_name}] resuming from epoch {start_epoch}')

    index_to_label = {index: label for label, index in repository.manifest['label_to_index'].items()}

    for epoch in range(start_epoch, EPOCHS + 1):
        model.train()
        train_loss_sum = 0.0
        train_count = 0
        for x_batch, y_batch in train_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            optimizer.zero_grad()
            logits = model(x_batch)
            loss = loss_fn(logits, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            train_loss_sum += float(loss.item()) * len(x_batch)
            train_count += len(x_batch)
        train_loss = train_loss_sum / max(1, train_count)

        val_metrics = evaluate_classifier(model, val_loader, val_subjects, loss_fn, index_to_label, device)
        history.append({
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': float(val_metrics['loss']),
            'val_accuracy': float(val_metrics['accuracy']),
            'val_macro_f1': float(val_metrics['macro_f1']),
        })
        print(
            f'[{experiment_name}] epoch={epoch:02d} '
            f'train_loss={train_loss:.4f} '
            f'val_loss={val_metrics["loss"]:.4f} '
            f'val_acc={val_metrics["accuracy"]:.4f} '
            f'val_macro_f1={val_metrics["macro_f1"]:.4f}'
        )

        if early_stopper.update(float(val_metrics['macro_f1']), epoch):
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}

        if SAVE_EVERY > 0 and (epoch % SAVE_EVERY == 0 or epoch == EPOCHS):
            save_checkpoint(latest_checkpoint_path, {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_model_state_dict': best_state,
                'early_stopper_state_dict': early_stopper.state_dict(),
                'history': history,
                'label_to_index': repository.manifest['label_to_index'],
                'channel_mode': channel_mode,
                'label_fraction': LABEL_FRACTION,
                'metadata': checkpoint_metadata(
                    experiment_name,
                    channel_mode,
                    train_split,
                    val_split,
                    test_split,
                    windows_root,
                    checkpoint_role='latest',
                    checkpoint_epoch=epoch,
                ),
            })

        if early_stopper.should_stop:
            print(f'[{experiment_name}] early stopping at epoch {epoch}')
            break

    if best_state is None:
        raise RuntimeError(f'[{experiment_name}] training completed without a best checkpoint')

    model.load_state_dict(best_state)
    train_metrics = evaluate_classifier(model, train_loader, train_subjects, loss_fn, index_to_label, device)
    val_metrics = evaluate_classifier(model, val_loader, val_subjects, loss_fn, index_to_label, device)
    test_metrics = evaluate_classifier(model, test_loader, test_subjects, loss_fn, index_to_label, device)

    stem = f'supervised_{channel_mode}_{experiment_name}_acc{metric_token(float(test_metrics["accuracy"]))}_macrof1{metric_token(float(test_metrics["macro_f1"]))}'
    checkpoint_path = output_dir / f'{stem}_checkpoint.pt'
    metrics_path = output_dir / f'{stem}_metrics.json'
    confusion_path = output_dir / f'{stem}_confusion_matrix.png'
    per_subject_path = output_dir / f'{stem}_per_subject_accuracy.csv'

    save_checkpoint(checkpoint_path, {
        'model_state_dict': model.state_dict(),
        'label_to_index': repository.manifest['label_to_index'],
        'channel_mode': channel_mode,
        'label_fraction': LABEL_FRACTION,
        'best_epoch': early_stopper.best_epoch,
        'metadata': checkpoint_metadata(experiment_name, channel_mode, train_split, val_split, test_split, windows_root),
    })

    metrics_payload = {
        'experiment_name': experiment_name,
        'best_epoch': early_stopper.best_epoch,
        'channel_mode': channel_mode,
        'label_fraction': LABEL_FRACTION,
        'train': {
            'loss': train_metrics['loss'],
            'accuracy': train_metrics['accuracy'],
            'macro_f1': train_metrics['macro_f1'],
        },
        'val': {
            'loss': val_metrics['loss'],
            'accuracy': val_metrics['accuracy'],
            'macro_f1': val_metrics['macro_f1'],
        },
        'test': {
            'loss': test_metrics['loss'],
            'accuracy': test_metrics['accuracy'],
            'macro_f1': test_metrics['macro_f1'],
            'per_class_f1': test_metrics['per_class_f1'],
            'per_subject_accuracy': test_metrics['per_subject_accuracy'],
        },
        'history': history,
        'artifacts': {
            'checkpoint_path': str(checkpoint_path),
            'latest_checkpoint_path': str(latest_checkpoint_path),
            'metrics_path': str(metrics_path),
            'confusion_path': str(confusion_path),
            'per_subject_accuracy_path': str(per_subject_path),
        },
    }

    metrics_path.write_text(json.dumps(metrics_payload, indent=2), encoding='utf-8')
    plot_confusion_matrix(test_metrics['confusion_matrix'], index_to_label, experiment_name, confusion_path)
    write_subject_accuracy(per_subject_path, test_metrics['per_subject_accuracy'])
    torch.cuda.empty_cache()
    return metrics_payload

In [ ]:
seed_everything(SEED)
device = require_cuda()
repository = WindowRepository(WINDOWS_ROOT)

results = []
for channel_mode in CHANNEL_MODES:
    print(f'\n=== Running {channel_mode} supervised baseline ===')
    payload = run_supervised_baseline(channel_mode, repository, device)
    results.append({
        'channel_mode': channel_mode,
        'run_name': RUN_NAMES[channel_mode],
        'experiment_name': EXPERIMENT_NAMES[channel_mode],
        'best_epoch': payload['best_epoch'],
        'val_accuracy': payload['val']['accuracy'],
        'val_macro_f1': payload['val']['macro_f1'],
        'test_accuracy': payload['test']['accuracy'],
        'test_macro_f1': payload['test']['macro_f1'],
        'metrics_path': payload['artifacts']['metrics_path'],
    })

results_df = pd.DataFrame(results)
display(results_df)

In [ ]:
summary_rows = []
for channel_mode in CHANNEL_MODES:
    output_dir = Path(OUTPUT_ROOT) / 'supervised' / RUN_NAMES[channel_mode]
    metrics_paths = sorted(output_dir.glob('*_metrics.json'))
    if not metrics_paths:
        continue
    payload = json.loads(metrics_paths[-1].read_text())
    summary_rows.append({
        'channel_mode': channel_mode,
        'run_name': RUN_NAMES[channel_mode],
        'experiment_name': payload['experiment_name'],
        'best_epoch': payload['best_epoch'],
        'val_accuracy': payload['val']['accuracy'],
        'val_macro_f1': payload['val']['macro_f1'],
        'test_accuracy': payload['test']['accuracy'],
        'test_macro_f1': payload['test']['macro_f1'],
        'metrics_path': str(metrics_paths[-1]),
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)